## Preparing the MITRE ATT&CK Retrieval Experiment

This notebook prepares the datasets for the retrieval experiments.

The project uses:

- `mitre_techniques.csv` as the retrieval corpus containing MITRE ATT&CK technique definitions.
- `tram_single_label.csv` as the query dataset containing cybersecurity threat-intelligence sentences with known ATT&CK technique labels.

The goals of this notebook are to:

1. Verify that TRAM ATT&CK labels exist in the MITRE ATT&CK v19.2 corpus.
2. Examine technique coverage and class distribution.
3. Remove examples that cannot be evaluated against the retrieval corpus.
4. Prepare the final query and corpus datasets for TF-IDF, BM25, and embedding retrieval.


### ATT&CK Version Compatibility

The TRAM single-label dataset initially contained 49 unique MITRE
ATT&CK technique identifiers. The labels were compared with the
Enterprise ATT&CK v19.2 corpus used as the retrieval knowledge base.

47 of the 49 TRAM technique identifiers were represented in the
retrieval corpus. T1562.001 and T1574.002 were not present in the
extracted corpus.

Examples associated with these two identifiers were excluded from
the evaluation. This was necessary because a retrieval system cannot
retrieve the correct technique when that technique is absent from
the search corpus.

The final evaluation dataset therefore contains 47 unique ATT&CK
techniques.

In [41]:
import pandas as pd

MITRE_PATH = "mitre_techniques.csv"
TRAM_PATH = "tram_single_label.csv"

mitre_df = pd.read_csv(MITRE_PATH)
tram_df = pd.read_csv(TRAM_PATH)

In [42]:
print("MITRE:", mitre_df.shape)
print("TRAM:", tram_df.shape)

MITRE: (697, 7)
TRAM: (3194, 4)


In [43]:
mitre_df.head()

,technique_id,name,description,is_subtechnique,platforms,tactics,stix_id
0,T1055.011,Extra Window Memory Injection,Adversaries may inject malicious code into pro...,True,['Windows'],"['stealth', 'privilege-escalation']",attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...
1,T1053.005,Scheduled Task,Adversaries may abuse the Windows Task Schedul...,True,['Windows'],"['execution', 'persistence', 'privilege-escala...",attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...
2,T1205.002,Socket Filters,Adversaries may attach filters to a network so...,True,"['Linux', 'macOS', 'Windows']","['stealth', 'persistence', 'command-and-control']",attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...
3,T1560.001,Archive via Utility,Adversaries may use utilities to compress and/...,True,"['Linux', 'macOS', 'Windows']",['collection'],attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...
4,T1021.005,VNC,Adversaries may use [Valid Accounts](https://a...,True,"['Linux', 'Windows', 'macOS']",['lateral-movement'],attack-pattern--01327cde-66c4-4123-bf34-5f258d...


In [44]:
tram_df.head()

,sentence,technique_id,doc_title,split
0,It spreads to Microsoft Windows machines using...,T1210,NotPetya Technical Analysis A Triple Threat F...,train
1,"Once a machine is infected by NotPetya, a seri...",T1570,NotPetya Technical Analysis A Triple Threat F...,train
2,Ransomware DLL C:\windows\perfc.dat,T1140,NotPetya Technical Analysis A Triple Threat F...,train
3,The malware decompresses its resource named 0x...,T1140,NotPetya Technical Analysis A Triple Threat F...,train
4,The malware is a DLL that is launched using ru...,T1218.011,NotPetya Technical Analysis A Triple Threat F...,train


In [45]:
print(mitre_df.columns.tolist())
print(tram_df.columns.tolist())

['technique_id', 'name', 'description', 'is_subtechnique', 'platforms', 'tactics', 'stix_id']
['sentence', 'technique_id', 'doc_title', 'split']


In [46]:
#missing values
print("MITRE missing values:")
print(
    mitre_df[
        ["technique_id", "name", "description"]
    ].isna().sum()
)

print("\nTRAM missing values:")
print(
    tram_df[
        ["sentence", "technique_id"]
    ].isna().sum()
)

MITRE missing values:
technique_id    0
name            0
description     0
dtype: int64

TRAM missing values:
sentence        0
technique_id    0
dtype: int64


In [47]:
#how many techniques does each contain
print(
    "MITRE techniques:",
    mitre_df["technique_id"].nunique()
)

print(
    "TRAM techniques:",
    tram_df["technique_id"].nunique()
)

MITRE techniques: 697
TRAM techniques: 49


In [48]:
#check 2 dataset compatibility
mitre_ids = set(
    mitre_df["technique_id"].dropna()
)

tram_ids = set(
    tram_df["technique_id"].dropna()
)

In [49]:
matching_ids = tram_ids & mitre_ids

print(
    "Matching TRAM techniques:",
    len(matching_ids)
)

Matching TRAM techniques: 47


In [50]:
missing_ids = tram_ids - mitre_ids

print(
    "TRAM techniques missing from MITRE:",
    len(missing_ids)
)

print(sorted(missing_ids))

TRAM techniques missing from MITRE: 2
['T1562.001', 'T1574.002']


In [51]:
coverage = (
    len(matching_ids) /
    len(tram_ids) * 100
)

print(
    f"TRAM technique compatibility: {coverage:.2f}%"
)

TRAM technique compatibility: 95.92%


In [52]:
excluded_ids = [
    "T1562.001",
    "T1574.002"
]
queries_df = tram_df[
    ~tram_df["technique_id"].isin(excluded_ids)
].copy()

In [53]:
print("Original examples:", len(tram_df))
print("Final usable examples:", len(queries_df))

print(
    "Unique techniques:",
    queries_df["technique_id"].nunique()
)

Original examples: 3194
Final usable examples: 3077
Unique techniques: 47


**removed 2 non-compatible technique from tram dataset**

In [54]:
#add technique name so the IDs are easier to understand
technique_name_map = (
    mitre_df
    .set_index("technique_id")["name"]
    .to_dict()
)

queries_df["technique_name"] = (
    queries_df["technique_id"]
    .map(technique_name_map)
)

In [55]:
queries_df[
    [
        "sentence",
        "technique_id",
        "technique_name",
        "split"
    ]
].head(10)

,sentence,technique_id,technique_name,split
0,It spreads to Microsoft Windows machines using...,T1210,Exploitation of Remote Services,train
1,"Once a machine is infected by NotPetya, a seri...",T1570,Lateral Tool Transfer,train
2,Ransomware DLL C:\windows\perfc.dat,T1140,Deobfuscate/Decode Files or Information,train
3,The malware decompresses its resource named 0x...,T1140,Deobfuscate/Decode Files or Information,train
4,The malware is a DLL that is launched using ru...,T1218.011,Rundll32,train
5,It is executed with the following arguments: #...,T1057,Process Discovery,train
6,The credential theft module is dropped if the ...,T1106,Native API,train
7,The malware decompresses a resource (0x1 if th...,T1140,Deobfuscate/Decode Files or Information,train
8,It calls OpenProcess on lsass.exe with access ...,T1003.001,LSASS Memory,train
9,"Server service, which both enforces security p...",T1003.001,LSASS Memory,train


In [56]:
print("Final examples:", len(queries_df))
print(
    "Unique techniques:",
    queries_df["technique_id"].nunique()
)

print("\nExamples per technique:")
print(
    queries_df["technique_id"]
    .value_counts()
    .describe()
)

Final examples: 3077
Unique techniques: 47

Examples per technique:
count     47.000000
mean      65.468085
std       81.898094
min        6.000000
25%       19.500000
50%       44.000000
75%       67.500000
max      467.000000
Name: count, dtype: float64


In [57]:
queries_df[
    ["technique_id", "technique_name"]
].value_counts().head(15)

technique_id  technique_name                         
T1027         Obfuscated Files or Information            467
T1140         Deobfuscate/Decode Files or Information    281
T1059.003     Windows Command Shell                      202
T1055         Process Injection                          197
T1105         Ingress Tool Transfer                      146
T1106         Native API                                 128
T1090         Proxy                                      114
T1078         Valid Accounts                             102
T1071.001     Web Protocols                               94
T1053.005     Scheduled Task                              78
T1003.001     LSASS Memory                                74
T1566.001     Spearphishing Attachment                    70
T1070.004     File Deletion                               65
T1204.002     Malicious File                              65
T1083         File and Directory Discovery                57
Name: count, dtype: int64

In [61]:
queries_df.to_csv(
    "tram_single_label_compatible.csv",
    index=False
)

print("Saved!")

Saved!


In [62]:
#final check
check_df = pd.read_csv(
    "tram_single_label_compatible.csv"
)

print("Rows:", len(check_df))
print(
    "Techniques:",
    check_df["technique_id"].nunique()
)

check_df.head()

Rows: 3077
Techniques: 47


,sentence,technique_id,doc_title,split,technique_name
0,It spreads to Microsoft Windows machines using...,T1210,NotPetya Technical Analysis A Triple Threat F...,train,Exploitation of Remote Services
1,"Once a machine is infected by NotPetya, a seri...",T1570,NotPetya Technical Analysis A Triple Threat F...,train,Lateral Tool Transfer
2,Ransomware DLL C:\windows\perfc.dat,T1140,NotPetya Technical Analysis A Triple Threat F...,train,Deobfuscate/Decode Files or Information
3,The malware decompresses its resource named 0x...,T1140,NotPetya Technical Analysis A Triple Threat F...,train,Deobfuscate/Decode Files or Information
4,The malware is a DLL that is launched using ru...,T1218.011,NotPetya Technical Analysis A Triple Threat F...,train,Rundll32
